
# What is the Ultralytics Library?

**Ultralytics** is a popular **PyTorch-based** library for training and using **YOLO** family models.

Main features:

- Easy installation and usage
- GPU support
- Training with only a few lines of code
- Simple Fine-tuning
- Support for different YOLO versions
- Built-in tools for Training, Validation, and Prediction

Many recent YOLO versions have been developed by the Ultralytics team.
```


In [ ]:
import os
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
import shutil

# Dataset Preparation

Before training, the dataset must be prepared according to the **YOLO annotation format**.

For each image, YOLO requires:

- One image file
- One corresponding `.txt` annotation file



In [ ]:
IMAGE_DIR = './dataset/images/'
ANNOTATIONS_DIR = './dataset/annotations/'
YOLO_DATASET_DIR = './Yolo_dataset'

In [ ]:
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images/val'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels/train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels/val'), exist_ok=True)

In [ ]:
class_names = [d for d in os.listdir(IMAGE_DIR) if os.path.isdir(os.path.join(IMAGE_DIR, d))]
class_names

In [ ]:
csv_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.csv')]
csv_files

In [ ]:
yolo_class_map = {name: i for i, name in enumerate(class_names)}
yolo_class_map



# YOLO Bounding Box Format

YOLO does not use corner coordinates.

Instead, each bounding box is represented as:

```

x_center
y_center
width
height

```

All values must be normalized between **0 and 1**.

---

## Converting from Corner Coordinates

If the dataset provides bounding boxes as:

```

x1
y1
x2
y2

```

they must be converted first.

### Center Coordinates

```

x_center = (x1 + x2) / 2

y_center = (y1 + y2) / 2

```

### Width and Height

```

width = x2 - x1

height = y2 - y1

```

### Normalization

```

x_center /= image_width

y_center /= image_height

width /= image_width

height /= image_height

```

After normalization, all bounding box values are in the range:

```

0 ≤ value ≤ 1

```

This format allows YOLO to handle images with different sizes consistently.
```


In [ ]:
dataset = []

for i in range(len(class_names)):
    class_name = class_names[i] 
    class_dir = os.path.join(IMAGE_DIR, class_name) 
    csv_file_name = csv_files[i] 
    
    csv_path = os.path.join(ANNOTATIONS_DIR, csv_file_name) 
    df_annotations = pd.read_csv(csv_path) 

    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name) 
        
        image = cv2.imread(image_path) 
        h, w, _ = image.shape 
        
        ann = df_annotations[df_annotations['image_name'] == image_name].iloc[0,1:].tolist()

        if (ann[2] > ann[0] and ann[3] > ann[1]): 
            x_min, y_min, x_max, y_max = [float(coord) for coord in ann]
            
            x_center = ((x_min + x_max) / 2) / w
            y_center = ((y_min + y_max) / 2) / h
            box_width = (x_max - x_min) / w
            box_height = (y_max - y_min) / h
            
            class_id = yolo_class_map[class_name] 
            yolo_label_content = f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}" 
            
            item_data = {
                'class_name': class_name,
                'image_name': image_name,
                'source_path': image_path,
                'yolo_label': yolo_label_content
            }
            dataset.append(item_data)
        else:
            print(f"️Invalid box found and removed in '{image_path}': {ann}") #

In [ ]:
train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42, stratify=[d['class_name'] for d in dataset])

In [ ]:
print(f"{len(train_data)}")
print(f"{len(val_data)}")

# Standard Dataset Structure

The YOLO dataset must follow a specific directory structure:

```text
dataset/

│
├── images/
│   ├── train/
│   └── val/
│
├── labels/
│   ├── train/
│   └── val/
````

The `images` folder contains the input images, while the `labels` folder contains the corresponding YOLO annotation files.

Each image must have a matching `.txt` label file with the same name.

Example:

```text
images/train/car01.jpg
labels/train/car01.txt
```

```
```


# Label File

For each image, a corresponding `.txt` label file is created.

Example:

```

0 0.512345 0.423421 0.351234 0.281245

```

The values are stored in this order:

```

Class_ID x_center y_center width height

```

If an image contains multiple objects, each object is stored in a separate line:

```

0 ...

2 ...

1 ...

0 ...

```

Each line represents one bounding box with its corresponding class and coordinates.
```


In [ ]:
def save_yolo_files(data_list, split_name):
    for item in data_list:
        class_name = item['class_name']
        image_name = item['image_name']
        source_image_path = item['source_path']
        yolo_label_content = item['yolo_label']
        
        new_unique_filename = f"{class_name}_{image_name}"
        base_new_filename = os.path.splitext(new_unique_filename)[0]

        dest_image_path = os.path.join(YOLO_DATASET_DIR, 'images', split_name, new_unique_filename)
        dest_label_path = os.path.join(YOLO_DATASET_DIR, 'labels', split_name, base_new_filename + '.txt')

        shutil.copy(source_image_path, dest_image_path)
        with open(dest_label_path, 'w') as f:
            f.write(yolo_label_content)

In [ ]:
save_yolo_files(train_data, 'train')

In [ ]:
save_yolo_files(val_data, 'val')

# Creating YAML File

Ultralytics only understands the entire Dataset by receiving the YAML file.

Example:

```yaml
path: dataset

train: images/train

val: images/val

nc: 3

names:

  0: airplane
  1: face
  2: motorcycle
````

---

## 1. Is this a dictionary?

This file:

```yaml
path: dataset

train: images/train

val: images/val

nc: 3

names:

  0: airplane
  1: face
  2: motorcycle
```

From the YAML structure perspective, yes, it is a **Mapping**, which is equivalent to a dictionary in Python.

If we read it in Python:

```python
import yaml

with open("data.yaml") as f:
    data = yaml.safe_load(f)

print(data)
```

Output:

```python
{
    'path': 'dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': {
        0: 'airplane',
        1: 'face',
        2: 'motorcycle'
    }
}
```

Therefore, `names` itself is also a dictionary.

---

## 2. Can we change the key names?

❌ No.

When Ultralytics reads the file, it looks exactly for these keys:

```yaml
path:
train:
val:
test:     # optional
nc:
names:
```

For example, if we write:

```yaml
dataset_path: dataset
```

or:

```yaml
training: images/train
```

or:

```yaml
number_of_classes: 3
```

the model does not recognize them and may ignore the file or produce an error.

---

## 3. Can we write `names` in another format?

Yes. There are two common formats.

### Method 1: Dictionary Format

```yaml
names:
  0: airplane
  1: face
  2: motorcycle
```

Equivalent to:

```python
{
    0: "airplane",
    1: "face",
    2: "motorcycle"
}
```

---

### Method 2: List Format

This format is more commonly recommended in Ultralytics:

```yaml
names:
  - airplane
  - face
  - motorcycle
```

or in one line:

```yaml
names: [airplane, face, motorcycle]
```

Equivalent to:

```python
["airplane", "face", "motorcycle"]
```

Ultralytics automatically assigns the indexes:

```
0 → airplane
1 → face
2 → motorcycle
```

```
```
